In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch_numopt
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import *
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from train_loop import train_loop

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
class Net(nn.Module):
    def __init__(self, input_size, device="cpu"):
        super().__init__()
        self.f1 = nn.Linear(input_size, 10, device=device)
        self.f2 = nn.Linear(10, 20, device=device)
        self.f3 = nn.Linear(20, 20, device=device)
        self.f4 = nn.Linear(20, 10, device=device)
        self.f5 = nn.Linear(10, 1, device=device)

        self.activation = nn.ReLU()
        # self.activation = nn.Sigmoid()

    def forward(self, x):
        x = self.activation(self.f1(x))
        x = self.activation(self.f2(x))
        x = self.activation(self.f3(x))
        x = self.activation(self.f4(x))
        x = self.f5(x)

        return x

In [4]:
# X, y = load_diabetes(return_X_y=True, scaled=False)
# X, y = make_regression(n_samples=1000, n_features=100)
X, y = make_friedman1(n_samples=1000, noise=1e-2)

X_scaler = MinMaxScaler()
X = X_scaler.fit_transform(X)
X = torch.Tensor(X).to(device)

y_scaler = MinMaxScaler()
y = y_scaler.fit_transform(y.reshape((-1, 1)))
y = torch.Tensor(y).to(device)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

torch_data = TensorDataset(X_train, y_train)
data_loader = DataLoader(torch_data, batch_size=10000)

torch.Size([800, 10]) torch.Size([800, 1])
torch.Size([200, 10]) torch.Size([200, 1])


In [5]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.LevenbergMarquardtLS(
    model=model,
    fletcher=False,
)

model, loss_history = train_loop(model, loss_fn, opt, data_loader, epochs=1000, max_patience=50)

epoch:  0, loss: 0.16902847588062286
epoch:  1, loss: 0.07252495735883713
epoch:  2, loss: 0.05255572125315666
epoch:  3, loss: 0.03818396478891373
epoch:  4, loss: 0.027939757332205772
epoch:  5, loss: 0.026724819093942642
epoch:  6, loss: 0.025308217853307724
epoch:  7, loss: 0.024990154430270195
epoch:  8, loss: 0.018738089129328728
epoch:  9, loss: 0.017180105671286583
epoch:  10, loss: 0.011794235557317734
epoch:  11, loss: 0.011307637207210064
epoch:  12, loss: 0.010884707793593407
epoch:  13, loss: 0.010486879386007786
epoch:  14, loss: 0.01006489247083664
epoch:  15, loss: 0.009693956933915615
epoch:  16, loss: 0.009299599565565586
epoch:  17, loss: 0.008970040827989578
epoch:  18, loss: 0.008629836142063141
epoch:  19, loss: 0.008302315138280392
epoch:  20, loss: 0.007979359477758408
epoch:  21, loss: 0.007630534004420042
epoch:  22, loss: 0.007270858157426119
epoch:  23, loss: 0.006900336593389511
epoch:  24, loss: 0.006507350131869316
epoch:  25, loss: 0.006093200296163559
e

In [6]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.9992009401321411
Test metrics:  R2 = 0.9981840252876282


In [7]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.LevenbergMarquardtLS(model=model, fletcher=True, solver="pinv")

model, loss_history = train_loop(model, loss_fn, opt, data_loader, epochs=1000, max_patience=50)

epoch:  0

/home/eugeniolr/Documents/repos/torch_numopt/src/torch_numopt/numerical_optimizer.py:152: UserWarning: torch.linalg.svd: During SVD computation with the selected cusolver driver, batches 0 failed to converge. A more accurate method will be used to compute the SVD as a fallback. Check doc at https://pytorch.org/docs/stable/generated/torch.linalg.svd.html (Triggered internally at /pytorch/aten/src/ATen/native/cuda/linalg/BatchLinearAlgebraLib.cpp:692.)
  if torch.linalg.cond(h) > 1e8:


, loss: 0.15890227258205414
epoch:  1, loss: 0.15886355936527252
epoch:  2, loss: 0.05301659554243088
epoch:  3, loss: 0.03417205438017845
epoch:  4, loss: 0.0298540610820055
epoch:  5, loss: 0.023427600041031837
epoch:  6, loss: 0.02279970608651638
epoch:  7, loss: 0.02039017342031002
epoch:  8, loss: 0.014755232259631157
epoch:  9, loss: 0.012434045784175396
epoch:  10, loss: 0.01241812203079462
epoch:  11, loss: 0.012128121219575405
epoch:  12, loss: 0.010908825322985649
epoch:  13, loss: 0.010001933202147484
epoch:  14, loss: 0.009997665882110596
epoch:  15, loss: 0.009060167707502842
epoch:  16, loss: 0.00840543769299984
epoch:  17, loss: 0.0078929103910923
epoch:  18, loss: 0.007370393257588148
epoch:  19, loss: 0.00705277593806386
epoch:  20, loss: 0.006848611868917942
epoch:  21, loss: 0.006700629834085703
epoch:  22, loss: 0.006694488227367401
epoch:  23, loss: 0.006329543422907591
epoch:  24, loss: 0.0061078742146492004
epoch:  25, loss: 0.005962947383522987
epoch:  26, loss:

In [8]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.9054688215255737
Test metrics:  R2 = 0.7978556752204895
